In [ ]:
!pip install pandas numpy matplotlib seaborn plotly wordcloud pyarrow

In [ ]:
# ── Install dependencies (uncomment if needed) ──────────────────────────────
# !pip install pandas numpy matplotlib seaborn plotly wordcloud pyarrow

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, re
from collections import Counter

warnings.filterwarnings('ignore')

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (10, 5),
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

PALETTE   = sns.color_palette('Set2')
LEVEL_CLR = {1: '#4CAF50', 2: '#FF9800', 3: '#F44336'}  # green / orange / red

print('Libraries loaded ✓')

In [ ]:
PATH = '../datasets/processed/processed_gaia.parquet'
df   = pd.read_parquet(PATH)

print(f'Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
df['query_len'] = df['query'].apply(len)
df['has_file'] = df['file_name'].apply(lambda x: 0 if x is None or x == "" else 1)

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
sns.countplot(data=df, x='level', ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('1. Query Distribution by Level', fontweight='bold')
axes[0, 0].set_xlabel('GAIA Level')

sns.boxplot(data=df, x='level', y='steps_num', ax=axes[0, 1], palette='magma')
axes[0, 1].set_title('2. Steps Required per Level', fontweight='bold')
axes[0, 1].set_ylabel('Number of Steps')


# 3. Complexity vs. Tool Usage (The 'Interaction' Proxy)
# This validates your VI (Interaction) variable
sns.barplot(data=df, x='level', y='tool_num', ax=axes[0, 2], palette='rocket', capsize=.1)
axes[0, 2].set_title('3. Tool Usage Intensity per Level', fontweight='bold')
axes[0, 2].set_ylabel('Avg Number of Tools')

# 4. Complexity vs. Latency (The 'Resource' Proxy)
# This justifies the need for a Router (TR) to save time
sns.violinplot(data=df, x='level', y='time_taken', ax=axes[1, 0], palette='crest')
axes[1, 0].set_title('4. Time Taken Distribution', fontweight='bold')
axes[1, 0].set_ylabel('Time (minutes)')

# 5. Correlation Heatmap
# To see how well steps, tools, and length predict complexity
corr_cols = ['level', 'steps_num', 'tool_num', 'time_taken', 'query_len', 'has_file']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", ax=axes[1, 1])
axes[1, 1].set_title('5. Feature Correlation Heatmap', fontweight='bold')

# 6. Scatter: Steps vs Time (Identifying the "Efficiency Frontier")
sns.scatterplot(data=df, x='steps_num', y='time_taken', hue='level', 
                style='level', s=100, ax=axes[1, 2], palette='viridis')
axes[1, 2].set_title('6. Steps vs. Time (Colored by Level)', fontweight='bold')

plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd

def get_ground_truth_modality(row):
    """
    Expert algorithmic assignment of complexity and modality.
    Based on user-defined tiered rules for AI execution patterns.
    
    Tiers ordered by execution complexity:
    - Tier 3: Multi-Agent Pipeline (steps > 15 OR tools >= 4)
    - Tier 2: Single Agent (tools in [1, 3])
    - Tier 1: Reasoning Workflow (tool_num == 0 AND steps > 4)
    - Tier 0: Direct LLM Call (tool_num == 0 AND steps <= 4)
    """
    T = pd.to_numeric(row.get('tool_num'), errors='coerce')
    S = pd.to_numeric(row.get('steps_num'), errors='coerce')
    t = pd.to_numeric(row.get('time_taken'), errors='coerce')

    # Replace missing values with 0 to avoid ambiguous pd.NA booleans in comparisons.
    T = 0 if pd.isna(T) else T
    S = 0 if pd.isna(S) else S
    t = 0 if pd.isna(t) else t

    # Tier 3: Multi-Agent Pipeline
    # Primary: steps_num > 15 OR tool_num >= 4
    # Supporting: time_taken > 30 mins
    if S > 15 or T >= 4:
        return pd.Series([3, 'MA'])

    # Tier 2: Single Agent
    # Primary: tool_num between [1, 3]
    # Supporting: steps_num between [5, 15]
    if 1 <= T <= 3:
        return pd.Series([2, 'SA'])

    # Tier 1: Reasoning Workflow
    # Primary: tool_num == 0 AND steps_num > 4
    # Supporting: time_taken > 5 mins
    if T == 0 and S > 5:
        return pd.Series([1, 'RW'])

    # Tier 0: Direct LLM Call
    # Primary: tool_num == 0 AND steps_num <= 4
    # Supporting: time_taken < 5 mins
    return pd.Series([0, 'L'])

# Execution
df[['complexity_level', 'ai_modality']] = df.apply(get_ground_truth_modality, axis=1)

In [ ]:
from pathlib import Path

required_cols = ['query', 'level', 'complexity_level', 'ai_modality']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f'Missing required columns: {missing_cols}')

processed_df = df[required_cols].copy()
processed_df.columns = ['QUERY', 'LEVEL', 'COMPLEXITY_LEVEL', 'AI_MODALITY']

out_dir = Path('../datasets/processed')
out_dir.mkdir(parents=True, exist_ok=True)

out_parquet = out_dir / 'gaia_processed_complexity.parquet'
out_csv = out_dir / 'gaia_processed_complexity.csv'

processed_df.to_parquet(out_parquet, index=False)
processed_df.to_csv(out_csv, index=False)

print(f'Wrote: {out_parquet.resolve()}')
print(f'Wrote: {out_csv.resolve()}')
processed_df.head()

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load processed dataset
processed_path = Path('../datasets/processed/gaia_processed_complexity.parquet')
proc = pd.read_parquet(processed_path)

print('Processed dataset loaded')
print(f'Shape: {proc.shape[0]:,} rows x {proc.shape[1]} columns')
display(proc.head())

# Basic quality checks
print('\nNull values per column:')
display(proc.isna().sum().to_frame('null_count'))

print('\nUnique values per label column:')
display(pd.DataFrame({
    'LEVEL_unique': [proc['LEVEL'].nunique(dropna=True)],
    'COMPLEXITY_LEVEL_unique': [proc['COMPLEXITY_LEVEL'].nunique(dropna=True)],
    'AI_MODALITY_unique': [proc['AI_MODALITY'].nunique(dropna=True)],
}))

# Frequency tables
level_dist = proc['LEVEL'].value_counts(dropna=False).sort_index()
complexity_dist = proc['COMPLEXITY_LEVEL'].value_counts(dropna=False).sort_index()
modality_dist = proc['AI_MODALITY'].value_counts(dropna=False)

print('\nLEVEL distribution:')
display(level_dist.to_frame('count').assign(percent=(level_dist / len(proc) * 100).round(2)))

print('\nCOMPLEXITY_LEVEL distribution:')
display(complexity_dist.to_frame('count').assign(percent=(complexity_dist / len(proc) * 100).round(2)))

print('\nAI_MODALITY distribution:')
display(modality_dist.to_frame('count').assign(percent=(modality_dist / len(proc) * 100).round(2)))

# Cross-tab analyses
lvl_vs_complex = pd.crosstab(proc['LEVEL'], proc['COMPLEXITY_LEVEL'])
lvl_vs_modality = pd.crosstab(proc['LEVEL'], proc['AI_MODALITY'])

print('\nLEVEL x COMPLEXITY_LEVEL (counts):')
display(lvl_vs_complex)

print('\nLEVEL x AI_MODALITY (counts):')
display(lvl_vs_modality)

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.countplot(data=proc, x='LEVEL', ax=axes[0], palette='viridis')
axes[0].set_title('Processed Data: LEVEL Distribution')

sns.countplot(data=proc, x='COMPLEXITY_LEVEL', ax=axes[1], palette='magma')
axes[1].set_title('Processed Data: COMPLEXITY_LEVEL Distribution')

sns.countplot(data=proc, x='AI_MODALITY', order=proc['AI_MODALITY'].value_counts().index, ax=axes[2], palette='Set2')
axes[2].set_title('Processed Data: AI_MODALITY Distribution')

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.heatmap((lvl_vs_complex.div(lvl_vs_complex.sum(axis=1), axis=0) * 100).round(1), 
            annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': '%'})
axes[0].set_title('LEVEL x COMPLEXITY_LEVEL (Row %)')

sns.heatmap((lvl_vs_modality.div(lvl_vs_modality.sum(axis=1), axis=0) * 100).round(1), 
            annot=True, fmt='.1f', cmap='Blues', ax=axes[1], cbar_kws={'label': '%'})
axes[1].set_title('LEVEL x AI_MODALITY (Row %)')

plt.tight_layout()
plt.show()

# Save analysis outputs
analysis_dir = Path('../datasets/processed/analysis')
analysis_dir.mkdir(parents=True, exist_ok=True)

level_dist.to_frame('count').assign(percent=(level_dist / len(proc) * 100).round(2)).to_csv(analysis_dir / 'level_distribution.csv')
complexity_dist.to_frame('count').assign(percent=(complexity_dist / len(proc) * 100).round(2)).to_csv(analysis_dir / 'complexity_distribution.csv')
modality_dist.to_frame('count').assign(percent=(modality_dist / len(proc) * 100).round(2)).to_csv(analysis_dir / 'modality_distribution.csv')
lvl_vs_complex.to_csv(analysis_dir / 'level_vs_complexity_counts.csv')
lvl_vs_modality.to_csv(analysis_dir / 'level_vs_modality_counts.csv')

print(f'\nAnalysis files saved to: {analysis_dir.resolve()}')

In [ ]:
PATH = '../datasets/processed/swe_bench.parquet'
df_swe   = pd.read_parquet(PATH)

print(f'Shape  : {df_swe.shape[0]:,} rows × {df_swe.shape[1]} columns')
print(f'Columns: {df_swe.columns.tolist()}')

In [ ]:
import ast
import re
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# Golden labeling rubric for SWE-Bench complexity + router modality
# Uses available columns only: difficulty, num_fail_to_pass, FAIL_TO_PASS, hints_text
# -----------------------------------------------------------------------------

def _norm_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip().lower()

def _count_tests(value):
    """Count tests listed in FAIL_TO_PASS robustly across list/str formats."""
    if pd.isna(value):
        return 0
    if isinstance(value, (list, tuple, set)):
        return len(value)

    s = str(value).strip()
    if not s:
        return 0

    # Try Python list literal first, e.g. "['a', 'b']"
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple, set)):
            return len(parsed)
    except Exception:
        pass

    # Fallback: split by common separators
    parts = [p for p in re.split(r"\n|,|;", s) if p.strip()]
    return len(parts)

# 1) Feature engineering from SWE-Bench metadata
swe = df_swe.copy()
swe['difficulty_norm'] = swe['difficulty'].apply(_norm_text)
swe['hint_present'] = swe['hints_text'].fillna('').astype(str).str.strip().ne('')
swe['f2p_num'] = pd.to_numeric(swe['num_fail_to_pass'], errors='coerce').fillna(0).clip(lower=0)
swe['failing_tests_count'] = swe['FAIL_TO_PASS'].apply(_count_tests)

# 2) Rule-based complexity score (transparent + explainable)
difficulty_weight = {
    'easy': 0.0,
    'medium': 1.0,
    'hard': 2.0
}
swe['difficulty_w'] = swe['difficulty_norm'].map(difficulty_weight).fillna(1.0)

# Score components (tunable):
swe['complexity_score'] = (
    swe['difficulty_w']
    + np.clip(swe['f2p_num'], 0, 8) * 0.35
    + np.clip(swe['failing_tests_count'], 0, 12) * 0.12
    + swe['hint_present'].astype(int) * 0.25
)

# 3) Golden complexity labels: 0..3
def score_to_level(score):
    if score < 1.2:
        return 0  # very low
    if score < 2.4:
        return 1  # low/moderate
    if score < 3.8:
        return 2  # high
    return 3      # very high

swe['COMPLEXITY_LEVEL'] = swe['complexity_score'].apply(score_to_level).astype(int)

# 4) Router modality labels (aligned with your GAIA taxonomy)
# L  : Direct LLM call
# RW : Reasoning workflow
# SA : Single-agent tool loop
# MA : Multi-agent pipeline
def level_to_modality(row):
    lvl = int(row['COMPLEXITY_LEVEL'])
    f2p = row['f2p_num']
    ntests = row['failing_tests_count']

    # Escalate to MA when iterative repair burden is clearly high
    if lvl == 3 and (f2p >= 6 or ntests >= 10):
        return 'MA'
    if lvl >= 2:
        return 'SA'
    if lvl == 1:
        return 'RW'
    return 'L'

swe['AI_MODALITY'] = swe.apply(level_to_modality, axis=1)

# 5) Final golden label table
golden_swe = swe[[
    'id',
    'query',
    'difficulty',
    'num_fail_to_pass',
    'failing_tests_count',
    'complexity_score',
    'COMPLEXITY_LEVEL',
    'AI_MODALITY'
]].copy()

print('Golden labels created for SWE-Bench')
print(f'Shape: {golden_swe.shape[0]:,} rows x {golden_swe.shape[1]} columns')
display(golden_swe.head())

print('\nCOMPLEXITY_LEVEL distribution:')
display(golden_swe['COMPLEXITY_LEVEL'].value_counts().sort_index().to_frame('count'))

print('\nAI_MODALITY distribution:')
display(golden_swe['AI_MODALITY'].value_counts().to_frame('count'))

# Optional: save output
from pathlib import Path
out_dir = Path('../datasets/processed')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'swe_bench_golden_labels.csv'
golden_swe.to_csv(out_path, index=False)
print(f'\nSaved: {out_path.resolve()}')